<div style="background: linear-gradient(135deg, #8338ec 0%, #3a0ca3 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">🚰 03 — Pipelines (Escudos Contra Data Leakage)</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 02 · Bloco 04 · Ajuste</p>
</div>

## 🎯 Objetivos

- Construir Pipelines sklearn profissionais
- Evitar **data leakage** no preprocessing
- Integrar Pipeline com GridSearchCV

---

## 1️⃣ O Problema: Data Leakage

Se você normaliza ANTES de dividir treino/teste, o scaler "vê" dados de teste.
Isso vaza informação e infla artificialmente suas métricas.

```python
# ERRADO!
scaler.fit_transform(X)  # Vê TUDO
X_train, X_test = split(X)

# CERTO!
X_train, X_test = split(X)
scaler.fit(X_train)      # Só vê treino
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)  # Usa parâmetros do treino
```

## 2️⃣ Pipeline: A Solução Elegante

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target

# Pipeline: scaler + modelo = tudo junto
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Cross-validation com Pipeline (scaler aprende SÓ no treino de cada fold!)
scores = cross_val_score(pipe, X, y, cv=5, scoring='f1')
print(f'F1-Score: {scores.mean():.4f} ± {scores.std():.4f}')
print('\n→ Sem data leakage! O scaler é re-fitado em cada fold automaticamente.')

## 3️⃣ Pipeline + GridSearchCV

In [ ]:
# Buscar melhores hiperparâmetros DENTRO do pipeline
param_grid = {
    'modelo__n_estimators': [50, 100, 200],
    'modelo__max_depth': [3, 5, 10, None],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X, y)

print(f'Melhor F1: {grid.best_score_:.4f}')
print(f'Melhores params: {grid.best_params_}')

## 4️⃣ Pipeline com Múltiplos Passos

In [ ]:
from sklearn.decomposition import PCA
from sklearn.svm import SVC

pipe_complexo = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=10)),
    ('svm', SVC(kernel='rbf'))
])

scores = cross_val_score(pipe_complexo, X, y, cv=5, scoring='accuracy')
print(f'Pipeline: Scaler → PCA(10) → SVM')
print(f'Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

## 🏋️ Questões

### Q1. Crie um pipeline com `PolynomialFeatures` + `Ridge`. Busque o melhor `degree` com GridSearch.

### Q2. Compare o score COM e SEM Pipeline (normalizando antes do split). A diferença é grande?

### Q3. Use `Pipeline` + `GridSearchCV` para testar 3 modelos diferentes no mesmo pipeline.

In [ ]:
# Resolva aqui
